<a href="https://colab.research.google.com/github/hedil1/ML-Project/blob/main/Modele_Finale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_excel("dataset_final.xlsx")

In [ ]:
import pandas as pd

df = pd.read_excel("dataset_final.xlsx")

# nettoyer colonnes
df.columns = df.columns.str.strip()

# créer target
df["Dangereux"] = ((df["Blessés"] > 0) | (df["Tués"] > 0)).astype(int)

# features
X = df.drop(["Dangereux", "Blessés", "Tués"], axis=1)
y = df["Dangereux"]

** split des données**

In [ ]:
#diviser le dataset en données d’entraînement et de test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Logistic Regression**

In [ ]:
#transformer les variables catégorielles en variables numériques grâce au One-Hot Encoding
X = pd.get_dummies(X)

In [ ]:
# créer target
df["Dangereux"] = ((df["Blessés"] > 0) | (df["Tués"] > 0)).astype(int)

# créer X & y
X = df.drop(["Dangereux", "Blessés", "Tués"], axis=1)
y = df["Dangereux"]

# encoder
X = pd.get_dummies(X)

# split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# voir combien de NaN il y a
print(X.isnull().sum().sort_values(ascending=False))

PK Ligne                    3
Date_2021-01-19             0
Date_2021-01-15             0
Date_2021-01-13             0
Date_2021-01-10             0
                           ..
Date_2019-03-20 00:00:00    0
Date_2019-03-16 00:00:00    0
Date_2019-03-14 00:00:00    0
Date_2019-03-11 00:00:00    0
Date_2019-04-26 00:00:00    0
Length: 396, dtype: int64


In [ ]:
#Corriger le Nan par median
X["PK Ligne"] = X["PK Ligne"].fillna(X["PK Ligne"].median())

In [ ]:
X = X.drop(columns=[col for col in X.columns if "Date_" in col]) #drop colone Date

In [ ]:
# voir combien de NaN il y a de nuveau
print(X.isnull().sum().sort_values(ascending=False))

N°                 0
Retard             0
Gare/PK            0
PK Ligne           0
UnitéAffaure       0
                  ..
Gouvernorat_SF     0
Gouvernorat_SL     0
Gouvernorat_SS     0
Gouvernorat_TN     0
Gouvernorat_UNK    0
Length: 150, dtype: int64


In [ ]:
# colonnes numériques
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# colonnes texte/catégorielles
cat_cols = X.select_dtypes(include=['object', 'bool']).columns

# remplacer NaN numériques par la médiane
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# remplacer NaN catégorielles par "Unknown"
for col in cat_cols:
    X[col] = X[col].fillna("Unknown")

In [ ]:
X = pd.get_dummies(X, drop_first=True)
print(X.isnull().sum().sum())

0


le modèle n’a pas complètement appris l’algorithme s’est arrêté avant de trouver la meilleure solution souvent à cause de :
1. données non normalisées
2.  trop de features
3.  valeurs très différentes

**On faire Normalisation**

In [ ]:
#standardiser les variables numériques pour les rendre comparables
from sklearn.preprocessing import StandardScaler  #StandardScaler un outil qui permet de normaliser les données.

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train) #calcule la moyenne et l’écart-type sur X_train
X_test = scaler.transform(X_test) #X_train devient centré et réduit

**Entraînement propre**

In [ ]:
#créer et entraîner un modèle de classification basé sur la régression logistique.
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000) #permet d’assurer la convergence du modèle
model.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

**Vérifier performance**




In [ ]:
#évaluer les performances du modèle de machine learning après son entraînement.
from sklearn.metrics import accuracy_score, classification_report
 #accuracy_score → précision globale
#classification_report → détail des performances
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9491525423728814
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        29
           1       1.00      0.90      0.95        30

    accuracy                           0.95        59
   macro avg       0.95      0.95      0.95        59
weighted avg       0.95      0.95      0.95        59



**Vérifier performance**

In [ ]:
#calcule la probabilité d’appartenir à la classe 1
y_proba = model.predict_proba(X_test)[:, 1]
#transforme les probabilités en classes (0/1)
y_pred_new = (y_proba > 0.4).astype(int)

In [ ]:
#corrige le déséquilibre des classes cad évite que le modèle favorise la classe majoritaire
from sklearn.linear_model import LogisticRegression
#modèle de classification
model = LogisticRegression(class_weight="balanced", max_iter=2000)

In [ ]:
#modèle basé sur plusieurs arbres de décision
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(max_depth=6, n_estimators=200)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=6, n_estimators=200)

**Cross-validation**
évalue le modèle de façon plus fiable que un simple split train/test
1. découpe les données en 5 parties
2. entraîne le modèle 5 fois
3. change le jeu de test à chaque fois

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)  #découpe les données en 5 parties & entraîne le modèle 5 fois

print(scores.mean())

0.9518994739918177


*changer* le variable dangereux Dangereux = 1 si :
Nbre d'intersection > 8
ET Piste > 3
ET Sécurité == 0
Sinon = 0

In [ ]:
#créer une variable cible
import numpy as np

df["Dangereux"] = np.where(
    (df["Nbre d'intersection"] > 8) &
    (df["Piste"] > 3) &
    (df["Sécurité"] == 0),
    1,
    0
)

In [ ]:
# Séparation X / y on drop ces doonees car éviter fuite de données (data leakage)
X = df.drop(["Dangereux", "Nbre d'intersection", "Piste", "Sécurité"], axis=1)
y = df["Dangereux"]

In [ ]:
#Traitement de la date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day

df = df.drop("Date", axis=1)

Apres cette etape j'essayer d echnegr le valeur enrte dans le descson de variavl dnagereurs car il etre a epeux pres 1 qui est pas tres logique dans un modele Ml

In [ ]:
#analyser le type réel des valeurs contenues dans chaque colonne du Data
for col in df.columns:
    print(col, df[col].apply(type).value_counts().head())

N° N°
<class 'float'>    291
Name: count, dtype: int64
Lieu Lieu
<class 'str'>    291
Name: count, dtype: int64
Lieu.Lieu Lieu.Lieu
<class 'str'>    291
Name: count, dtype: int64
Retard Retard
<class 'float'>    291
Name: count, dtype: int64
Gare/PK Gare/PK
<class 'float'>    291
Name: count, dtype: int64
PK Ligne PK Ligne
<class 'float'>    291
Name: count, dtype: int64
Blessés Blessés
<class 'int'>    291
Name: count, dtype: int64
Tués Tués
<class 'int'>    291
Name: count, dtype: int64
Ligne Ligne
<class 'str'>    291
Name: count, dtype: int64
Zone Zone
<class 'str'>    291
Name: count, dtype: int64
UnitéAffaure UnitéAffaure
<class 'float'>    291
Name: count, dtype: int64
UnitéAffaire UnitéAffaire
<class 'str'>    291
Name: count, dtype: int64
Sécurité Sécurité
<class 'float'>    291
Name: count, dtype: int64
Mois Mois
<class 'str'>    291
Name: count, dtype: int64
Gouvernorat Gouvernorat
<class 'str'>    291
Name: count, dtype: int64
Nbre d'intersection Nbre d'intersection
<class 

In [ ]:
#CRÉATION DE LA VARIABLE CIBLE
df["Dangereux"] = df["Dangereux"].astype(int)

In [ ]:
cols_to_drop = [
    "Dangereux",
    "Blessés",
    "Tués"
]

In [ ]:
X = df.drop(columns=cols_to_drop)


Mlagre modele fonctionne propormeent mais logistic regression n'a pas assez itere car les donnes n'est pas bien standaralisees **texte en gras**

**Apres la standascaler le modee stabailise logistic resgression**

In [ ]:
#entraîner un modèle de Machine Learning
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = df["Dangereux"]
X = df.drop(columns=["Dangereux", "Blessés", "Tués"])


#Séparation automatique des types de colonnes
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

#Remplace valeurs manquantes & Normalise les valeurs
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#Remplace valeurs manquantes & Transforme texte → nombres
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

#nettoyage & transformation & apprentissage

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=5000))
])

#Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Entraînement
model.fit(X_train, y_train)

#Prédiction
y_pred = model.predict(X_test)
#Accuracy → % de bonnes prédictions
#Precision / Recall → qualité du modèle


print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9322033898305084
              precision    recall  f1-score   support

           0       0.89      0.96      0.92        25
           1       0.97      0.91      0.94        34

    accuracy                           0.93        59
   macro avg       0.93      0.94      0.93        59
weighted avg       0.93      0.93      0.93        59



Test de CROOS VALIDATION

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print("CV Accuracy:", scores.mean()) #MOYENNE DES SCORES
print("Std:", scores.std())

CV Accuracy: 0.8862653419053185
Std: 0.07990409691855187


**DummyClassifier**

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print(dummy.score(X_test, y_test))

0.576271186440678


CV Accuracy = 0.886 (≈ 88.6%) Std = 0.0799 (≈ 8%) le modele tourne autour de ~88% de précision réelle Écart-type est 0;0799 cad les résultats changent selon le split le modèle est pas totalement stable

Le modèle atteint une accuracy moyenne de 88.6% en validation croisée avec un écart-type de 0.08, ce qui indique une performance globalement stable mais légèrement sensible au découpage des données.

In [ ]:
# RandomForestClassifier
#un modèle basé sur des arbres de décision multiples
from sklearn.ensemble import RandomForestClassifier

#preprocessing (imputation + encoding + scaling)
rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=200, #modèle plus robuste
        random_state=42
    ))
])

scores_rf = cross_val_score(rf, X, y, cv=5) #tester sur modèle sur 5 splits différents du dataset
#resultat
print("Random Forest CV Accuracy:", scores_rf.mean())
print("Std:", scores_rf.std())

Random Forest CV Accuracy: 0.9655756867329048
Std: 0.028885488916112277


In [ ]:
#GradientBoostingClassifier
#modèle basé sur le principe de boosting  Il corrige les erreurs des modèles précédents
from sklearn.ensemble import GradientBoostingClassifier

#Random Forest = plusieurs arbres indépendants
#Gradient Boosting = arbres construits séquentiellement
gb = Pipeline([
    ("prep", preprocess),
    ("clf", GradientBoostingClassifier(random_state=42))
])
#Validation croisée
scores_gb = cross_val_score(gb, X, y, cv=5)

print("Gradient Boosting CV Accuracy:", scores_gb.mean())
print("Std:", scores_gb.std())

Gradient Boosting CV Accuracy: 0.9896551724137932
Std: 0.013793103448275851


In [ ]:
#XGBClassifier
from xgboost import XGBClassifier
#XGBoost = version optimisée de Gradient Boosting

xgb = Pipeline([
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=300, #300 arbres
        learning_rate=0.05,
        max_depth=4, #apprentissage lent mais stable
        random_state=42,
        eval_metric="logloss" #limite la complexité → évite overfitting
    ))
])


#Validation croisée
scores_xgb = cross_val_score(xgb, X, y, cv=5)

print("XGBoost CV Accuracy:", scores_xgb.mean())
print("Std:", scores_xgb.std())

XGBoost CV Accuracy: 0.9828170660432496
Std: 0.018887346453783576


In [ ]:
# SVM = Support Vector Machine
#un algorithme de classification puissant basé sur les marges
from sklearn.svm import SVC

#permet de capturer des relations non linéaires complexes
svm = Pipeline([
    ("prep", preprocess),
    ("clf", SVC(kernel="rbf"))
])

#teste le modèle sur 5 découpages différents
scores_svm = cross_val_score(svm, X, y, cv=5)

print("SVM CV Accuracy:", scores_svm.mean())
print("Std:", scores_svm.std())

SVM CV Accuracy: 0.9035067212156633
Std: 0.04453229133567767


In [ ]:
#KNeighborsClassifier
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    ("prep", preprocess),
    ("clf", KNeighborsClassifier(n_neighbors=5))
])

scores_knn = cross_val_score(knn, X, y, cv=5)

print("KNN CV Accuracy:", scores_knn.mean())
print("Std:", scores_knn.std())

KNN CV Accuracy: 0.8208065458796024
Std: 0.07624137572384693


In [ ]:
#Comparaison
results = {
    "Random Forest": scores_rf.mean(),
    "Gradient Boosting": scores_gb.mean(),
    "XGBoost": scores_xgb.mean(),
    "SVM": scores_svm.mean(),
    "KNN": scores_knn.mean()
}

for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(k, ":", v)

Gradient Boosting : 0.9896551724137932
XGBoost : 0.9828170660432496
Random Forest : 0.9655756867329048
SVM : 0.9035067212156633
KNN : 0.8208065458796024


Target imbalance check :
1.  dataset équilibré
2. accuracy est fiable
3.  pas de biais fort de classe



In [ ]:
df["Dangereux"].value_counts(normalize=True)

,proportion
Dangereux,
1,0.546392
0,0.453608


In [ ]:
import joblib

joblib.dump(gb, "GradientBoosting.pkl")
joblib.dump(xgb, "XGBoost.pkl")
joblib.dump(rf, "RandomForest.pkl")
joblib.dump(svm, "SVM.pkl")
joblib.dump(knn, "KNN.pkl")

['KNN.pkl']

In [ ]:
from google.colab import files

files.download("GradientBoosting.pkl")
files.download("XGBoost.pkl")
files.download("RandomForest.pkl")
files.download("SVM.pkl")
files.download("KNN.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>